In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger

# RDKit'in terminali dolduran kırmızı uyarılarını kapatıyoruz
RDLogger.DisableLog('rdApp.*')

# Kullanılacak donanım
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sistem hazır. Kullanılan işlem birimi: {device}")

# --- ONCOMIND GRAPHVAE AYARLARI ---
MAX_ATOMS = 40  # Lösemi ilaçlarında genelde atom sayısı 40'ı geçmez. Her molekülü 40x40 matrislere sabitleyeceğiz.
ATOM_TYPES = ['C', 'N', 'O', 'F', 'P', 'S', 'Cl', 'Br', 'I'] # İzin verdiğimiz atom türleri
NUM_ATOM_TYPES = len(ATOM_TYPES)

Sistem hazır. Kullanılan işlem birimi: cuda


In [25]:
def smiles_to_matrices(smiles):
    mol = Chem.MolFromSmiles(smiles)

    # Molekül bozuksa veya 40 atomdan büyükse atla
    if mol is None or mol.GetNumAtoms() > MAX_ATOMS:
        return None, None

    # 1. BAĞ MATRİSİ (A) - 40x40
    A = np.zeros((MAX_ATOMS, MAX_ATOMS), dtype=np.float32)
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        # Bağ türünü sayıya çeviriyoruz (Tekli: 1.0, Aromatik: 1.5, İkili: 2.0, Üçlü: 3.0)
        bond_type = bond.GetBondTypeAsDouble()
        A[i, j] = A[j, i] = bond_type

    # 2. ATOM MATRİSİ (X) - 40x9 (One-Hot Encoding)
    X = np.zeros((MAX_ATOMS, NUM_ATOM_TYPES), dtype=np.float32)
    for i, atom in enumerate(mol.GetAtoms()):
        symbol = atom.GetSymbol()
        if symbol in ATOM_TYPES:
            X[i, ATOM_TYPES.index(symbol)] = 1.0

    return torch.tensor(A), torch.tensor(X)

class LeukemiaGraphDataset(Dataset):
    def __init__(self, txt_file):
        with open(txt_file, 'r') as f:
            raw_smiles = [line.strip() for line in f.readlines()]

        self.A_data = []
        self.X_data = []

        print("Moleküller Matris formatına dönüştürülüyor...")
        for smi in raw_smiles:
            A, X = smiles_to_matrices(smi)
            if A is not None:
                self.A_data.append(A)
                self.X_data.append(X)

    def __len__(self):
        return len(self.A_data)

    def __getitem__(self, idx):
        return self.A_data[idx], self.X_data[idx]

# Hazırladığımız temiz dosyayı yüklüyoruz
dataset = LeukemiaGraphDataset('leukemia_jtvae_ready.txt')
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
print(f"Toplam {len(dataset)} adet ilaç adayı matris formatında eğitime hazır!")

Moleküller Matris formatına dönüştürülüyor...
Toplam 26016 adet ilaç adayı matris formatında eğitime hazır!


In [26]:
class OncoMindGraphVAE(nn.Module):
    def __init__(self, max_atoms, num_atom_types, latent_dim=128):
        super(OncoMindGraphVAE, self).__init__()
        self.max_atoms = max_atoms
        self.num_atom_types = num_atom_types

        # Giriş boyutu: (40x40 = 1600) + (40x9 = 360) = Toplam 1960 piksel
        input_dim = (max_atoms * max_atoms) + (max_atoms * num_atom_types)

        # --- ENCODER (Veriyi Sıkıştıran Kısım) ---
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU()
        )
        self.mu_layer = nn.Linear(512, latent_dim)
        self.logvar_layer = nn.Linear(512, latent_dim)

        # --- DECODER (Yoktan Var Eden Kısım) ---
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU()
        )

        # Çıkış Katmanları
        self.decode_A = nn.Linear(1024, max_atoms * max_atoms)
        self.decode_X = nn.Linear(1024, max_atoms * num_atom_types)

    def reparameterize(self, mu, logvar):
        # Uzayda rastgelelik yaratan "Z" hesaplaması
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, A, X):
        batch_size = A.size(0)

        # Matrisleri tek bir uzun çizgi (vektör) haline getiriyoruz
        A_flat = A.view(batch_size, -1)
        X_flat = X.view(batch_size, -1)
        inputs = torch.cat([A_flat, X_flat], dim=1)

        # Zihne yaz (Encode)
        hidden = self.encoder(inputs)
        mu = self.mu_layer(hidden)
        logvar = self.logvar_layer(hidden)

        # Hayal et (Latent Space Z)
        z = self.reparameterize(mu, logvar)

        # Çiz (Decode)
        dec_hidden = self.decoder(z)

        # Çıktıları orijinal matris şekillerine geri döndürüyoruz
        # Bağlar 0 ile ~3 arasında olduğu için ReLu (negatif olmasın diye) kullanıyoruz
        A_pred = F.relu(self.decode_A(dec_hidden)).view(batch_size, self.max_atoms, self.max_atoms)

        # Elementler (X) olasılık olduğu için Softmax kullanıyoruz
        X_pred = torch.softmax(self.decode_X(dec_hidden).view(batch_size, self.max_atoms, self.num_atom_types), dim=-1)

        return A_pred, X_pred, mu, logvar

# Modeli belleğe al
model = OncoMindGraphVAE(MAX_ATOMS, NUM_ATOM_TYPES, latent_dim=128).to(device)

In [27]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def vae_loss(A_pred, A_real, X_pred, X_real, mu, logvar):
    # 1. Yeniden İnşa Hatası (Reconstruction Loss)
    # Model orijinal A ve X matrislerine ne kadar benziyor?
    loss_A = F.mse_loss(A_pred, A_real, reduction='sum')
    loss_X = F.mse_loss(X_pred, X_real, reduction='sum')

    # 2. Hayal Gücü Dengesi (KL Divergence)
    # Latent uzayın pürüzsüz kalmasını sağlar
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return loss_A + loss_X + kl_loss

EPOCHS = 10 # Geliştirme aşamasında isen 50'ye kadar çıkarabilirsin
print("Eğitim Başlıyor...")

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for A_batch, X_batch in dataloader:
        A_batch = A_batch.to(device)
        X_batch = X_batch.to(device)

        optimizer.zero_grad()

        A_pred, X_pred, mu, logvar = model(A_batch, X_batch)

        loss = vae_loss(A_pred, A_batch, X_pred, X_batch, mu, logvar)
        loss.backward()

        # Gradyan patlamalarını engelle
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Ortalama Kayıp (Loss): {avg_loss:.4f}")

torch.save(model.state_dict(), 'oncomind_graphvae.pth')
print("Model başarıyla eğitildi ve kaydedildi!")

Eğitim Başlıyor...
Epoch 01/10 | Ortalama Kayıp (Loss): 82.2093
Epoch 02/10 | Ortalama Kayıp (Loss): 80.6625
Epoch 03/10 | Ortalama Kayıp (Loss): 79.6849
Epoch 04/10 | Ortalama Kayıp (Loss): 79.0185
Epoch 05/10 | Ortalama Kayıp (Loss): 78.8333
Epoch 06/10 | Ortalama Kayıp (Loss): 78.6126
Epoch 07/10 | Ortalama Kayıp (Loss): 77.2860
Epoch 08/10 | Ortalama Kayıp (Loss): 77.1713
Epoch 09/10 | Ortalama Kayıp (Loss): 76.9419
Epoch 10/10 | Ortalama Kayıp (Loss): 76.6332
Model başarıyla eğitildi ve kaydedildi!


In [28]:
def matrices_to_smiles(A, X):
    """
    Modelin ürettiği A (Bağ) ve X (Atom) matrislerini SMILES'a çevirir.
    Eğer kimyasal olarak imkansız bir molekül üretilmişse 'None' döner.
    """
    # RDKit'in düzenlenebilir molekül (RWMol) objesini oluşturuyoruz
    mol = Chem.RWMol()

    # Hangi atomların gerçekten var olduğunu (Boş/Padding olmayanları) buluyoruz
    node_features = np.argmax(X, axis=1) # Örn: [0, 0, 1...] -> Oksijen
    active_atoms = [] # Geriye kalan boşlukları (sıfırları) atlamak için liste

    # 1. ATOMLARI EKLE
    for i in range(MAX_ATOMS):
        # A matrisinde bu atoma ait bağların toplamı 0'dan büyükse, bu atom "aktif"tir
        if np.sum(A[i, :]) > 0.1:
            atom_idx = node_features[i]
            atom_symbol = ATOM_TYPES[atom_idx]
            new_atom = Chem.Atom(atom_symbol)
            idx = mol.AddAtom(new_atom)
            active_atoms.append((i, idx))

    # 2. BAĞLARI EKLE
    for matrix_i, mol_i in active_atoms:
        for matrix_j, mol_j in active_atoms:
            if matrix_i >= matrix_j: # Matris simetrik olduğu için yarısını okumak yeterli
                continue

            bond_val = A[matrix_i, matrix_j]
            # Modelin tahmin ettiği değeri en yakın bağ türüne yuvarlıyoruz
            if bond_val > 0.5 and bond_val < 1.2:
                mol.AddBond(mol_i, mol_j, Chem.BondType.SINGLE)
            elif bond_val >= 1.2 and bond_val < 1.8:
                mol.AddBond(mol_i, mol_j, Chem.BondType.AROMATIC)
            elif bond_val >= 1.8 and bond_val < 2.5:
                mol.AddBond(mol_i, mol_j, Chem.BondType.DOUBLE)
            elif bond_val >= 2.5:
                mol.AddBond(mol_i, mol_j, Chem.BondType.TRIPLE)

    try:
        # Kimyasal kuralları (Değerlikler vs.) kontrol et (Sanitization)
        Chem.SanitizeMol(mol)
        smiles = Chem.MolToSmiles(mol)
        return smiles if smiles != "" else None
    except Exception:
        return None # Model Karbona 6 bağ çizmişse RDKit çökmez, None döneriz.

print("Matris->SMILES çevirici fonksiyonu aktif!")

Matris->SMILES çevirici fonksiyonu aktif!


In [29]:
model.eval()

NUM_SAMPLES = 1000 # Uzaydan 1000 farklı rastgele aday deneyeceğiz
valid_smiles_generated = []

print(f"{NUM_SAMPLES} adet rastgele uzay vektöründen molekül üretiliyor...")

with torch.no_grad():
    # Latent uzay (Z) için rastgele gürültü yaratıyoruz
    z_random = torch.randn(NUM_SAMPLES, 128).to(device)

    # Decoder'a gürültüyü verip yeni matrisleri üretiyoruz
    dec_hidden = model.decoder(z_random)
    A_gen = F.relu(model.decode_A(dec_hidden)).view(NUM_SAMPLES, MAX_ATOMS, MAX_ATOMS).cpu().numpy()
    X_gen = torch.softmax(model.decode_X(dec_hidden).view(NUM_SAMPLES, MAX_ATOMS, NUM_ATOM_TYPES), dim=-1).cpu().numpy()

    # Üretilen 1000 matrisi SMILES'a çevirmeye çalışıyoruz
    for i in range(NUM_SAMPLES):
        smi = matrices_to_smiles(A_gen[i], X_gen[i])
        if smi is not None:
            valid_smiles_generated.append(smi)

# Benzersiz (unique) olanları filtreliyoruz
valid_smiles_generated = list(set(valid_smiles_generated))
validity_rate = (len(valid_smiles_generated) / NUM_SAMPLES) * 100

print("\n--- ÜRETİM SONUÇLARI ---")
print(f"Toplam Deneme: {NUM_SAMPLES}")
print(f"Kimyasal Olarak Geçerli & Sentezlenebilir Molekül: {len(valid_smiles_generated)}")
print(f"Modelin Geçerlilik (Validity) Oranı: %{validity_rate:.1f}\n")

print("Örnek 5 Yeni İlaç Adayı (De Novo):")
for i, smi in enumerate(valid_smiles_generated[:5]):
    print(f"{i+1}. {smi}")

1000 adet rastgele uzay vektöründen molekül üretiliyor...

--- ÜRETİM SONUÇLARI ---
Toplam Deneme: 1000
Kimyasal Olarak Geçerli & Sentezlenebilir Molekül: 3
Modelin Geçerlilik (Validity) Oranı: %0.3

Örnek 5 Yeni İlaç Adayı (De Novo):
1. C.C.C.CCCCCCCCCCCC.CCCCCCCCCCCCCCCCCCCC
2. C.C.C.CCCCCCCCCCCCCC.CCCCCCCCCCCCCCCCCCCC
3. C.C.CCCCCCCCCC.CCCCCCCCCCCCCCCCCCCC
